# CSC2541

End-to-end pipeline: download ChEMBL molecules → featurize → generate analogs via PrexSyn → score results.

Each section can be run independently if intermediate files already exist.

In [1]:
import pathlib
import sys

# ── resolve project root ───────────────────────────────────────────────────────
_root = pathlib.Path.cwd()
if not (_root / "src").exists():
    _root = _root.parent
for _p in ["src/utils", "src/generation", "src/evaluation"]:
    sys.path.insert(0, str(_root / _p))

# ── paths ──────────────────────────────────────────────────────────────────────
DATA_DIR         = _root / "data"
CHEMBL_FULL_CSV  = DATA_DIR / "chembl_full.csv"
CHEMBL_CSV       = DATA_DIR / "chembl_sampled.csv"
CHEMBL_CACHE     = DATA_DIR / "chembl_35_chemreps.txt.gz"
FEATURES_NPZ     = DATA_DIR / "chembl_features.npz"
SAMPLES_JSON     = DATA_DIR / "prexsyn_sampled.json"
SCORES_CSV       = DATA_DIR / "chembl_scores.csv"
AIZYNTH_DIR      = DATA_DIR / "aizynthfinder"

# ── pipeline settings ─────────────────────────────────────────────────────────
NUM_MOLECULES = 100
SEED          = 42
NUM_SAMPLES   = 64
API_URL       = "http://100.65.172.100:8011/sample"
LIMIT         = None       # set to e.g. 10 for a quick test run

## 1. Download ChEMBL

Downloads the full ChEMBL release, filters for valid drug-like organic molecules, and saves the complete set.  
This file is large (~1.8M molecules) but only needs to be downloaded once — reuse it across experiments by changing `SEED` or `NUM_MOLECULES` in the next cell.  
Skipped automatically if the output file already exists.

In [2]:
from download_chembl import download

if CHEMBL_FULL_CSV.exists():
    print(f"Already exists, skipping download: {CHEMBL_FULL_CSV}")
else:
    download(output=CHEMBL_FULL_CSV, cache=CHEMBL_CACHE)

Already exists, skipping download: d:\AI4DD Project\CSC2541-Project\data\chembl_full.csv


## 2. Sample from ChEMBL

Randomly samples `NUM_MOLECULES` from the full filtered set.  
Change `SEED` or `NUM_MOLECULES` above and re-run this cell to get a different subset — no need to re-download.

In [3]:
from download_chembl import sample as sample_chembl

sample_chembl(input=CHEMBL_FULL_CSV, output=CHEMBL_CSV, num_molecules=NUM_MOLECULES, seed=SEED)

Loaded 1,794,749 molecules from d:\AI4DD Project\CSC2541-Project\data\chembl_full.csv
Sampled 100 molecules → d:\AI4DD Project\CSC2541-Project\data\chembl_sampled.csv


WindowsPath('d:/AI4DD Project/CSC2541-Project/data/chembl_sampled.csv')

## 3. Featurize

Computes ECFP4, FCFP4, RDKit descriptors, and BRICS fragment fingerprints for each molecule.  
Output is a `.npz` file with arrays shaped `(N, ...)`.  
Skipped if the output file already exists.

In [4]:
from featurize_chembl import featurize

featurize(input=CHEMBL_CSV, output=FEATURES_NPZ)

Loaded 100 molecules from d:\AI4DD Project\CSC2541-Project\data\chembl_sampled.csv


Featurizing: 100%|██████████| 100/100 [00:00<00:00, 171.54it/s]

Featurized 100 molecules.
Saved features to: d:\AI4DD Project\CSC2541-Project\data\chembl_features.npz


WindowsPath('d:/AI4DD Project/CSC2541-Project/data/chembl_features.npz')

In [5]:
# Inspect the feature arrays
import numpy as np

data = np.load(FEATURES_NPZ, allow_pickle=True)
for key in ["smiles", "ecfp4", "fcfp4", "rdkit_desc_values", "brics_fps", "brics_exists"]:
    arr = data[key]
    print(f"  {key:25s}: shape={arr.shape}  dtype={arr.dtype}")

  smiles                   : shape=(100,)  dtype=object
  ecfp4                    : shape=(100, 2048)  dtype=float32
  fcfp4                    : shape=(100, 2048)  dtype=float32
  rdkit_desc_values        : shape=(100, 43)  dtype=float32
  brics_fps                : shape=(100, 8, 2048)  dtype=float32
  brics_exists             : shape=(100, 8)  dtype=bool


## 4. Generate Analogs

Sends each molecule's features to the PrexSyn `/sample` API (QuerySampler with product-of-experts over all 5 properties).  
Make sure the Docker container is running: `docker start prexsyn`  
Set `LIMIT` above to a small number (e.g. 10) for a quick test.

In [6]:
import urllib.request

# Check API health before running
try:
    resp = urllib.request.urlopen("http://100.65.172.100:8011/health", timeout=5)
    print("API status:", resp.read().decode())
except Exception as e:
    print(f"API not reachable: {e}\nStart with: docker start prexsyn")

API status: {"status":"ok"}


In [7]:
from sampler import run_batch

run_batch(
    npz=FEATURES_NPZ,
    output=SAMPLES_JSON,
    url=API_URL,
    num_samples=NUM_SAMPLES,
    limit=LIMIT,
)

Processing 1000 molecules → http://100.65.172.100:8011/sample


  4%|▍         | 39/1000 [00:52<21:45,  1.36s/it] 


KeyboardInterrupt: 

## 5. Score Generated Molecules

Scores every generated SMILES against its source molecule:
- **Tanimoto** (ECFP4) — structural similarity
- **Desirability** — QED × Lipinski penalty × MW penalty × rotatable bonds penalty
- **Hit** — similar (≥ 0.3) AND drug-like (≥ 0.2) AND novel (< 1.0 Tanimoto)

In [ ]:
import json
from scoring import score_results, summarize

results = json.loads(SAMPLES_JSON.read_text())
print(f"Loaded {len(results)} source molecules")
print(f"Total generated: {sum(len(r.get('generated_smiles', [])) for r in results)}")

df = score_results(results, similarity_threshold=0.3, desirability_threshold=0.2, novelty_threshold=1.0)
df.to_csv(SCORES_CSV, index=False)
print(f"Saved {len(df)} scored rows to {SCORES_CSV}")

## 6. Results

In [ ]:
# Overall statistics
df[["tanimoto", "qed", "desirability", "is_hit", "is_similar", "is_drug_like", "is_novel"]].describe().round(3)

In [ ]:
# Hit rate summary per source molecule (top 10)
summary = summarize(df)
summary.head(10)

In [ ]:
total_hits = int(df["is_hit"].sum())
print(f"Total hits : {total_hits} / {len(df)} ({100 * total_hits / len(df):.1f}%)")
print(f"Hit rate   : {df['is_hit'].mean():.3f}")
print(f"Mean Tanimoto : {df['tanimoto'].mean():.3f}")
print(f"Mean QED      : {df['qed'].mean():.3f}")
print(f"Unique scaffolds: {df['scaffold'].nunique()}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

df["tanimoto"].hist(bins=40, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].axvline(0.3, color="red", linestyle="--", label="threshold (0.3)")
axes[0].set_title("Tanimoto Similarity")
axes[0].set_xlabel("Tanimoto")
axes[0].legend()

df["qed"].hist(bins=40, ax=axes[1], color="seagreen", edgecolor="white")
axes[1].set_title("QED (Drug-likeness)")
axes[1].set_xlabel("QED")

df["desirability"].hist(bins=40, ax=axes[2], color="darkorange", edgecolor="white")
axes[2].axvline(0.2, color="red", linestyle="--", label="threshold (0.2)")
axes[2].set_title("Desirability Score")
axes[2].set_xlabel("Desirability")
axes[2].legend()

plt.tight_layout()
plt.savefig(DATA_DIR / "score_distributions.png", dpi=150)
plt.show()

## 7. Reproduce Paper Metrics

Reconstruction rate and mean Tanimoto similarity reported in the PrexSyn paper (sample=64).

In [ ]:
import json
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import rdMolDescriptors

def ecfp4(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)

# Load raw API output (unfiltered)
results = json.loads(SAMPLES_JSON.read_text())

all_sims      = []   # every generated molecule's Tanimoto
max_tanimotos = []   # best per source molecule
reconstructed = 0

for entry in results:
    src_fp = ecfp4(entry["source_smiles"])
    if src_fp is None:
        continue

    sims = []
    for gen_smi in entry.get("generated_smiles", []):
        gen_fp = ecfp4(gen_smi)
        if gen_fp is None:
            continue
        sim = DataStructs.TanimotoSimilarity(src_fp, gen_fp)
        sims.append(sim)
        all_sims.append(sim)

    if not sims:
        continue

    best = max(sims)
    max_tanimotos.append(best)
    if best == 1.0:
        reconstructed += 1

recon_rate = reconstructed / len(max_tanimotos)

print(f"Molecules evaluated  : {len(max_tanimotos)}")
print(f"Samples per molecule : {NUM_SAMPLES}")
print(f"Total generated      : {len(all_sims)}")
print()
print(f"Recons. %            : {100 * recon_rate:.1f}%")
print(f"Similarity (max/src) : {np.mean(max_tanimotos):.3f}   ← mean of best per source")
print(f"Similarity (all)     : {np.mean(all_sims):.3f}   ← mean of all generated")
print(f"                                         (paper target: 0.73)")

## 8. Synthesizability Filter (AiZynthFinder)

Uses AiZynthFinder retrosynthesis to evaluate whether generated molecules are synthesizable.  
Models are loaded once and cached for the entire session.

In [ ]:
import subprocess

AIZYNTH_DIR.mkdir(parents=True, exist_ok=True)

if not (AIZYNTH_DIR / "uspto_model.onnx").exists():
    print(f"Downloading AiZynthFinder models to {AIZYNTH_DIR} ...")
    subprocess.run(["download_public_data", str(AIZYNTH_DIR)], check=True)
    print("Done.")
else:
    print(f"Models already present: {AIZYNTH_DIR}")

In [ ]:
import importlib
import synth_parallel
importlib.reload(synth_parallel)

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm

# ── test molecules ─────────────────────────────────────────────────────────────
test_molecules = {
    "1":  "Cc1cc(C(F)(F)F)nc(Nc2ccccc2C(=O)NCc2ccccc2Cl)n1",
    "2":  "Cc1cc(C(F)(F)F)nc(Nc2ccccc2C(=O)c2ccccc2Cl)n1",
    "3":  "Cc1c(Cl)cccc1Nc1cc(C(F)(F)F)nc(Nc2ccccc2)n1",
    "4":  "COC(=O)c1ccccc1Nc1nc(Nc2cccc(C(F)(F)F)c2C)cc(C(F)(F)F)n1",
    "5":  "COC(=O)c1cccc(Nc2nc(Cc3cccc(Cl)c3C)cc(C(F)(F)F)n2)c1",
    "6":  "Cc1ccccc1COc1ccccc1Nc1nc(C(F)(F)F)cc(C(F)(F)F)n1",
    "7":  "CC(=O)c1ccccc1Nc1nc(Cc2ccccc2C)cc(C(F)(F)F)n1",
    "8":  "FC(F)(F)COc1cc(C(F)(F)F)nc(Nc2c(Cl)cccc2Cl)n1",
    "9":  "O=C(CCc1ccccc1Cl)Nc1ccccc1Nc1nc(C(F)(F)F)cc(C(F)(F)F)n1",
    "10": "Cc1cccc(C)c1C(=O)c1ccccc1Nc1nc(C(F)(F)F)cc(C(F)(F)F)n1",
    "11": "CCc1ccccc1Nc1nc(-c2ccccc2Cl)cc(C(F)(F)F)n1",
    "12": "Cc1c(Nc2nc(C(F)F)cc(C(F)(F)F)n2)cccc1C(=O)NCC(C)C",
    "13": "Cc1cc(C(F)(F)F)nc(Nc2ccccc2Cl)n1",
    "14": "Cc1ccccc1C(=O)c1ccccc1Nc1nc(C(F)(F)F)cc(C(F)(F)F)n1",
    "15": "O=C(Nc1nc(C(F)(F)F)cc(C(F)(F)F)n1)c1ccccc1Nc1c(Cl)cccc1Cl",
    "16": "CS(=O)(=O)c1ccccc1CNC(=O)c1ccccc1Nc1nc(C(F)(F)F)cc(C(F)(F)F)n1",
    "17": "Cc1ccccc1Nc1nc(CCCc2ccccc2Cl)cc(C(F)(F)F)n1",
    "18": "CCOC(=O)C(Cc1ccccc1Cl)c1cc(C(F)(F)F)nc(Nc2c(Cl)cccc2Cl)n1",
    "19": "FC(F)(F)c1cc(C(F)(F)F)nc(Nc2ccccc2Cl)n1",
    "20": "CNC(=O)c1ccccc1Nc1nc(C(F)(F)F)cc(C(F)(F)F)n1",
}

# ── parallel scoring ───────────────────────────────────────────────────────────
N_WORKERS = 4   # adjust to available CPU cores
config    = str(AIZYNTH_DIR / "config.yml")

results_map: dict[str, dict] = {}

with ProcessPoolExecutor(
    max_workers = N_WORKERS,
    initializer = synth_parallel.worker_init,
    initargs    = (config,),
) as executor:
    futures = {
        executor.submit(synth_parallel.score_one_full, smi): name
        for name, smi in test_molecules.items()
    }
    with tqdm(total=len(futures), desc="AiZynthFinder", unit="mol") as pbar:
        for future in as_completed(futures):
            name = futures[future]
            results_map[name] = future.result()
            pbar.update(1)

# ── print results (original order) ────────────────────────────────────────────
print(f"\n{'Molecule':<15} {'Solved':<8} {'Routes':<8} {'Top Score'}")
print("-" * 45)
for name in test_molecules:
    res = results_map[name]
    print(f"{name:<15} {str(res['is_solved']):<8} {res['num_routes']:<8} {res['top_score']:.3f}")